# Energy Prices — Ingestion & Cleaning

## Objective
Parse the GME (Gestore Mercati Energetici) day-ahead market archive into a single
hourly price series for the NORD bidding zone, then clean and lightly feature-engineer
it for the capture-rate and hedge models downstream.

- **Source**: GME MGP (Mercato del Giorno Prima / day-ahead market) published XML
  archive, 2005-01-01 to 2026-03-22, one file per calendar day.
- **NORD** is the Northern Italy bidding zone (where the plant modelled in this
  thesis sits); **PUN** (Prezzo Unico Nazionale) is the single national reference
  price, kept for context and zonal-spread analysis.
- GME publishes in **Italian local time (Europe/Rome)**, so a small number of
  calendar days carry 23 or 25 hourly records around the March/October DST
  transitions — handled explicitly in **Decision D1** below.

## Pipeline Position

| Inputs | This notebook | Outputs |
|---|---|---|
| `Data/Raw/MGP_Prezzi2005010120260322.zip` (GME XML archive) | Parse → clean → engineer | `Data/Cleaned/df_energy_cleaned.parquet` |
| | | `Data/Interim/GME_MGP_NORD_2005_2026.csv` (parsed checkpoint) |

## Setup

In [1]:
import sys
import zipfile
import xml.etree.ElementTree as ET
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd


def find_project_root() -> Path:
    """Walk up from the cwd until the directory holding Code/ and Data/ is found.

    Notebooks have no __file__, and a bare Path("..") is cwd-relative, so it
    breaks whenever the notebook is run from anywhere but its own folder.
    """
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "Code").is_dir() and (cand / "Data").is_dir():
            return cand
    raise RuntimeError(f"Project root not found above {here}")


ROOT = find_project_root()
INPUT_ZIP    = ROOT / "Data" / "Raw"     / "MGP_Prezzi2005010120260322.zip"
INTERIM_CSV  = ROOT / "Data" / "Interim" / "GME_MGP_NORD_2005_2026.csv"
OUTPUT_CLEAN = ROOT / "Data" / "Cleaned" / "df_energy_cleaned.parquet"

LOCAL_TZ = "Europe/Rome"

## Parse the GME Archive

Each XML file holds one calendar day of hourly `<Prezzi>` records, with `Ora`
a 1-based settlement-hour index (23 on the spring-forward DST day, 25 on the
fall-back day, 24 otherwise).

In [2]:
def parse_italian_float(value: str) -> float:
    """Convert an Italian-formatted number string (comma decimal, dot thousands) to float."""
    if not value or value.strip() == "":
        return float("nan")
    return float(value.strip().replace(".", "").replace(",", "."))


def parse_xml_file(xml_content: bytes, source_name: str = "") -> list:
    """Parse a single GME XML file into a list of hourly records.

    # D1: Ora is a 1-based settlement-hour index for the calendar date in
    # Italian local time. Anchoring at local midnight and converting to UTC
    # *before* adding the hour offset keeps every settlement hour a distinct,
    # regularly-spaced UTC instant — 23 on the spring-forward day, 25 on the
    # fall-back day — with no DST gap, duplicate, or midnight overflow.
    """
    records = []
    try:
        root = ET.fromstring(xml_content)
    except ET.ParseError as e:
        print(f"  [SKIP] XML parse error in {source_name}: {e}")
        return records

    current_date, local_midnight_utc = None, None
    for elem in root.findall("Prezzi"):
        try:
            raw_date = elem.findtext("Data", "").strip()   # YYYYMMDD
            raw_ora  = elem.findtext("Ora", "").strip()    # 1-based
            raw_nord = elem.findtext("NORD", "").strip()
            raw_pun  = elem.findtext("PUN", "").strip()

            if not raw_date or not raw_ora:
                continue

            if raw_date != current_date:
                current_date = raw_date
                base_date = datetime.strptime(raw_date, "%Y%m%d")
                local_midnight_utc = pd.Timestamp(base_date).tz_localize(LOCAL_TZ).tz_convert("UTC")

            ora_0based = int(raw_ora) - 1
            dt_utc = local_midnight_utc + pd.Timedelta(hours=ora_0based)

            records.append({
                "datetime_utc":      dt_utc,
                "date":              base_date.date(),
                "price_NORD_EURMWh": parse_italian_float(raw_nord),
                "price_PUN_EURMWh":  parse_italian_float(raw_pun),
            })

        except (ValueError, TypeError) as e:
            print(f"  [WARN] Row error in {source_name}: {e}")
            continue

    return records

In [3]:
all_records = []
with zipfile.ZipFile(INPUT_ZIP, "r") as zf:
    xml_names = sorted(n for n in zf.namelist() if n.lower().endswith(".xml"))
    print(f"Found {len(xml_names):,} XML files")
    for i, name in enumerate(xml_names):
        with zf.open(name) as f:
            all_records.extend(parse_xml_file(f.read(), source_name=name))
        if (i + 1) % 2000 == 0:
            print(f"  Processed {i + 1:,} / {len(xml_names):,} files...")

print(f"\nTotal raw records parsed: {len(all_records):,}")

Found 7,924 XML files


  Processed 2,000 / 7,924 files...


  Processed 4,000 / 7,924 files...


  Processed 6,000 / 7,924 files...



Total raw records parsed: 186,024


In [4]:
df = (
    pd.DataFrame(all_records)
    .sort_values("datetime_utc")
    .drop_duplicates(subset=["datetime_utc"], keep="first")
    .reset_index(drop=True)
)

n_expected = len(all_records)
n_kept = len(df)
print(f"Records after dedup on datetime_utc: {n_kept:,}  (dropped {n_expected - n_kept:,})")
print(f"Range: {df['datetime_utc'].min()}  ->  {df['datetime_utc'].max()}")

Records after dedup on datetime_utc: 186,024  (dropped 0)
Range: 2004-12-31 23:00:00+00:00  ->  2026-03-22 22:00:00+00:00


### Verify the timeline is DST-clean

A regular hourly UTC index should now have **zero** gaps introduced by the
March/October transitions — any remaining gap is a genuine hole in the archive,
not a DST artifact.

In [5]:
full_index_utc = pd.date_range(df["datetime_utc"].min(), df["datetime_utc"].max(), freq="h", tz="UTC")
missing = full_index_utc.difference(df["datetime_utc"])
print(f"Expected hourly steps : {len(full_index_utc):,}")
print(f"Missing steps         : {len(missing):,}")
if len(missing):
    print(f"First 10 gaps: {list(missing[:10])}")

Expected hourly steps : 186,024
Missing steps         : 0


## Save the Parsed Checkpoint

Parsing 7,924 XML files takes a while — this checkpoint lets the cleaning steps
below be re-run without re-parsing the archive.

In [6]:
INTERIM_CSV.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(INTERIM_CSV, index=False)
print(f"[SAVED] {INTERIM_CSV}  ({len(df):,} rows)")

[SAVED] C:\Users\LucasMonero\OneDrive - EloGroup\Documentos\data projects\Master Thesis\Project\Data\Interim\GME_MGP_NORD_2005_2026.csv  (186,024 rows)


## Data Quality Audit

What to look for: how many zero and extreme prices exist, and whether they
cluster in a way that points to a data artifact rather than a market outcome.

In [7]:
df = df.set_index("datetime_utc")

zeros = df[df["price_NORD_EURMWh"] == 0]
extremes = df[df["price_NORD_EURMWh"] > 500]

print(f"Zero NORD prices     : {len(zeros)}")
print(zeros.groupby(zeros.index.year).size().rename("count").to_string())
print(f"\nExtreme NORD prices (> 500 EUR/MWh) : {len(extremes)}")
print(extremes.groupby(extremes.index.year).size().rename("count").to_string())
print(f"\nMax price: {df['price_NORD_EURMWh'].max():.2f} EUR/MWh at {df['price_NORD_EURMWh'].idxmax()}")

Zero NORD prices     : 33
datetime_utc
2005    14
2013     4
2020     5
2025    10

Extreme NORD prices (> 500 EUR/MWh) : 891
datetime_utc
2021     13
2022    878

Max price: 871.00 EUR/MWh at 2022-08-29 17:00:00+00:00


> **Takeaway.** All 14 zero-price hours before 2010 sit on two isolated 2005
> dates (Jan 1 and Jul 10) where PUN is inconsistent — present but at a
> different magnitude on Jan 1, entirely absent on Jul 10 — while NORD alone
> reads a flat zero. That is a first-year zonal-feed reporting gap, not a
> cleared price. Every zero from 2010 onward co-occurs mostly with PUN also
> clearing at zero, consistent with the well-documented rise of near-zero
> midday prices as solar penetration grew, so those are kept as real. Extreme
> prices are concentrated in 2021-2022 (the European energy crisis) and are
> real market events, kept unchanged.

## Cleaning

### Decision Log

| # | Decision | Alternatives | Rationale | Impact |
|---|---|---|---|---|
| D1 | Anchor settlement hours at local midnight, convert to UTC, then offset | Naive hour-offset from midnight (previous approach) | Naive offset silently drops real hours on fall-back days (colliding with the next day's midnight) and leaves a permanent gap on spring-forward days | Removes DST-induced gaps/duplicates at the source; no interpolation needed to patch them |
| D2 | Treat NORD=0 as missing only before 2010 | Treat all zeros as missing; treat all zeros as valid | The 14 pre-2010 zeros are reporting gaps (see audit above); zeros from 2010 on mostly co-occur with PUN=0, a real market outcome | 14 values -> NaN then interpolated; ~9,700 later zeros kept as observed |
| D3 | Keep extreme prices (> 500 EUR/MWh), flag only | Cap/winsorize | They are real 2021-2022 crisis events the capture-rate model needs to see | No information loss, flag lets downstream models condition on regime |
| D4 | Linear interpolation, `limit=3` hours | Forward-fill; drop | Missing runs are short (isolated hours); a 3-hour cap avoids papering over longer real outages | Fills isolated NaNs, leaves any longer gap genuinely missing |

In [8]:
# D2: pre-2010 zeros are a reporting gap (see audit above), not a market outcome
ZERO_ARTIFACT_CUTOFF = 2010
df["zero_flag"] = df["price_NORD_EURMWh"] == 0
df.loc[df["zero_flag"] & (df.index.year <= ZERO_ARTIFACT_CUTOFF), "price_NORD_EURMWh"] = np.nan

# D3: extreme prices are real crisis-era events, flagged not removed
df["price_flag"] = 0
df.loc[df["price_NORD_EURMWh"] > 500, "price_flag"] = 2

In [9]:
# D4: isolated missing hours only, capped at 3h so longer outages stay missing
before_mask = df["price_NORD_EURMWh"].isna()

df["price_NORD_EURMWh"] = df["price_NORD_EURMWh"].interpolate(method="linear", limit=3, limit_direction="both")
df["price_PUN_EURMWh"]  = df["price_PUN_EURMWh"].ffill(limit=3)

df["imputed_flag"] = before_mask & df["price_NORD_EURMWh"].notna()

print(f"Missing before imputation : {before_mask.sum()}")
print(f"Missing after imputation  : {df['price_NORD_EURMWh'].isna().sum()}")
print(f"Records imputed           : {df['imputed_flag'].sum()}")

Missing before imputation : 14
Missing after imputation  : 0
Records imputed           : 14


In [10]:
# Spike flag: above the 95th percentile within its own year, excluding already-extreme hours
p95_year = df.groupby(df.index.year)["price_NORD_EURMWh"].transform(lambda x: x.quantile(0.95))
df.loc[(df["price_NORD_EURMWh"] > p95_year) & (df["price_flag"] == 0), "price_flag"] = 1

print(df.groupby("price_flag").size().rename({0: "normal", 1: "spike (>P95/yr)", 2: "extreme (>500)"}))

price_flag
normal             176311
spike (>P95/yr)      8822
extreme (>500)        891
dtype: int64


## Feature Engineering

Local-time calendar features (market behaviour follows the local clock), plus
the one rolling feature the capture-rate model actually reads.

In [11]:
df["datetime_local"] = df.index.tz_convert(LOCAL_TZ)
df["date"]  = df["datetime_local"].dt.date
df["hour"]  = df["datetime_local"].dt.hour

daily_mean = df.groupby("date")["price_NORD_EURMWh"].transform("mean")
df["daily_mean"] = daily_mean

df = df.reset_index().rename(columns={"datetime_utc": "datetime_utc"})
df = df[[
    "datetime_utc", "datetime_local", "date", "hour",
    "price_NORD_EURMWh", "price_PUN_EURMWh",
    "zero_flag", "imputed_flag", "price_flag", "daily_mean",
]]
df.head()

,datetime_utc,datetime_local,date,hour,price_NORD_EURMWh,price_PUN_EURMWh,zero_flag,imputed_flag,price_flag,daily_mean
0,2004-12-31 23:00:00+00:00,2005-01-01 00:00:00+01:00,2005-01-01,0,27.4,27.650421,False,False,0,22.74375
1,2005-01-01 00:00:00+00:00,2005-01-01 01:00:00+01:00,2005-01-01,1,23.7,24.088555,False,False,0,22.74375
2,2005-01-01 01:00:00+00:00,2005-01-01 02:00:00+01:00,2005-01-01,2,24.4,24.499444,False,False,0,22.74375
3,2005-01-01 02:00:00+00:00,2005-01-01 03:00:00+01:00,2005-01-01,3,23.5,23.960878,False,False,0,22.74375
4,2005-01-01 03:00:00+00:00,2005-01-01 04:00:00+01:00,2005-01-01,4,24.4,24.467464,False,False,0,22.74375


## Save & Handoff

### Conclusion
- Hourly NORD/PUN price series, 2005-01-01 to 2026-03-22, DST-clean in UTC.
- Zero-price reporting gaps and short outages imputed and flagged; extreme and
  spike hours flagged, not altered.

**Next notebook can rely on:** `Data/Cleaned/df_energy_cleaned.parquet`,
keyed on `datetime_utc`, for the capture-rate model
(`Code/Modelling/Capture Price Model.ipynb`) and the hedge notebook
(`Code/Simulation/05_hedge_contract.ipynb`).

In [12]:
assert df["datetime_utc"].is_monotonic_increasing and not df["datetime_utc"].duplicated().any()
assert df["price_NORD_EURMWh"].isna().sum() == 0

OUTPUT_CLEAN.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(OUTPUT_CLEAN, index=False)
print(f"[SAVED] {OUTPUT_CLEAN}")
print(f"  Final shape: {df.shape}")

[SAVED] C:\Users\LucasMonero\OneDrive - EloGroup\Documentos\data projects\Master Thesis\Project\Data\Cleaned\df_energy_cleaned.parquet
  Final shape: (186024, 10)
